In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('viridis')
%matplotlib inline

# Ignorar advertencias
import warnings
warnings.filterwarnings('ignore')

## 1. Cargar y Explorar los Datos

In [ ]:
# Crear un dataset sintético de precios de casas
# Basado en el clásico dataset de Boston Housing con características similares
np.random.seed(42)

n_samples = 506

# Generar características
data = {
    'CRIM': np.random.exponential(3, n_samples),  # Tasa de criminalidad
    'RM': np.random.normal(6.3, 0.7, n_samples),  # Número promedio de habitaciones
    'AGE': np.random.uniform(0, 100, n_samples),  # Proporción de casas antiguas
    'DIS': np.random.gamma(2, 2, n_samples),  # Distancia a centros de empleo
    'TAX': np.random.normal(400, 150, n_samples),  # Tasa de impuestos
    'PTRATIO': np.random.normal(18, 2, n_samples),  # Ratio alumno-maestro
    'LSTAT': np.random.gamma(3, 3, n_samples)  # % población de bajo estatus
}

df = pd.DataFrame(data)

# Generar variable objetivo (precio) con relación a las características
df['PRICE'] = (
    35  # Precio base
    - 0.5 * df['CRIM']  # Criminalidad reduce precio
    + 5.0 * df['RM']  # Más habitaciones aumenta precio
    - 0.05 * df['AGE']  # Casas más antiguas reducen precio
    - 1.0 * df['DIS']  # Mayor distancia reduce precio
    - 0.01 * df['TAX']  # Más impuestos reduce precio
    - 0.5 * df['PTRATIO']  # Mayor ratio reduce precio
    - 0.4 * df['LSTAT']  # Mayor % bajo estatus reduce precio
    + np.random.normal(0, 3, n_samples)  # Ruido aleatorio
)

# Asegurar precios positivos
df['PRICE'] = df['PRICE'].clip(lower=5)

print("Dimensiones del dataset:", df.shape)
print("\nPrimeras filas:")
df.head()

In [ ]:
# Descripción de las características
print("Descripción de las características:\n")
print("CRIM: Tasa de criminalidad per cápita")
print("RM: Número promedio de habitaciones por vivienda")
print("AGE: Proporción de unidades construidas antes de 1940")
print("DIS: Distancia ponderada a centros de empleo")
print("TAX: Tasa de impuesto a la propiedad")
print("PTRATIO: Ratio alumno-maestro")
print("LSTAT: % de población de bajo estatus socioeconómico")
print("PRICE: Valor medio de viviendas (miles de dólares)")

In [ ]:
# Estadísticas descriptivas
print("Estadísticas descriptivas:")
df.describe()

In [ ]:
# Verificar valores nulos
print("Valores nulos por columna:")
print(df.isnull().sum())

In [ ]:
# Visualizar la distribución del precio
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(df['PRICE'], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Precio (miles de $)')
plt.ylabel('Frecuencia')
plt.title('Distribución de Precios de Casas')

plt.subplot(1, 2, 2)
plt.boxplot(df['PRICE'])
plt.ylabel('Precio (miles de $)')
plt.title('Boxplot de Precios')

plt.tight_layout()
plt.show()

print(f"Precio medio: ${df['PRICE'].mean():.2f}k")
print(f"Precio mediano: ${df['PRICE'].median():.2f}k")
print(f"Rango de precios: ${df['PRICE'].min():.2f}k - ${df['PRICE'].max():.2f}k")

In [ ]:
# Matriz de correlación
plt.figure(figsize=(10, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Matriz de Correlación de Características')
plt.tight_layout()
plt.show()

# Mostrar correlaciones con el precio
print("\nCorrelación con el precio:")
print(correlation_matrix['PRICE'].sort_values(ascending=False))

## 2. Preparar los Datos

In [ ]:
# Separar características y variable objetivo
X = df.drop('PRICE', axis=1)
y = df['PRICE']

# Dividir en conjunto de entrenamiento y prueba (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Datos de entrenamiento: {X_train.shape[0]} muestras")
print(f"Datos de prueba: {X_test.shape[0]} muestras")
print(f"\nCaracterísticas: {list(X.columns)}")

## 3. Entrenar el Modelo de Regresión Lineal

In [ ]:
# Crear el modelo de regresión lineal
lr_model = LinearRegression()

# Entrenar el modelo
lr_model.fit(X_train, y_train)

print("Modelo entrenado exitosamente")
print(f"\nIntercepto (β₀): {lr_model.intercept_:.4f}")
print("\nCoeficientes (β₁, β₂, ...):")
for feature, coef in zip(X.columns, lr_model.coef_):
    print(f"  {feature}: {coef:.4f}")

## 4. Realizar Predicciones

In [ ]:
# Hacer predicciones en ambos conjuntos
y_train_pred = lr_model.predict(X_train)
y_test_pred = lr_model.predict(X_test)

# Mostrar algunas predicciones
print("Ejemplos de predicciones en el conjunto de prueba:")
comparacion = pd.DataFrame({
    'Precio Real': y_test[:10].values,
    'Precio Predicho': y_test_pred[:10],
    'Error': y_test[:10].values - y_test_pred[:10]
})
print(comparacion)

## 5. Evaluar el Modelo

In [ ]:
# Calcular métricas de evaluación
# Conjunto de entrenamiento
train_r2 = r2_score(y_train, y_train_pred)
train_mse = mean_squared_error(y_train, y_train_pred)
train_rmse = np.sqrt(train_mse)
train_mae = mean_absolute_error(y_train, y_train_pred)

# Conjunto de prueba
test_r2 = r2_score(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("Métricas de Evaluación:")
print("\nConjunto de Entrenamiento:")
print(f"  R² Score: {train_r2:.4f}")
print(f"  MSE: {train_mse:.4f}")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE: {train_mae:.4f}")

print("\nConjunto de Prueba:")
print(f"  R² Score: {test_r2:.4f}")
print(f"  MSE: {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE: {test_mae:.4f}")

print("\nInterpretación:")
print(f"  El modelo explica el {test_r2*100:.2f}% de la varianza en los precios")
print(f"  Error promedio absoluto: ${test_mae:.2f}k")

## 6. Visualizar Resultados

In [ ]:
# Gráfico de valores reales vs predichos
plt.figure(figsize=(12, 5))

# Conjunto de entrenamiento
plt.subplot(1, 2, 1)
plt.scatter(y_train, y_train_pred, alpha=0.5, edgecolors='k', linewidth=0.5)
plt.plot([y_train.min(), y_train.max()], [y_train.min(), y_train.max()], 'r--', lw=2)
plt.xlabel('Precio Real (miles de $)')
plt.ylabel('Precio Predicho (miles de $)')
plt.title(f'Conjunto de Entrenamiento\nR² = {train_r2:.4f}')
plt.grid(alpha=0.3)

# Conjunto de prueba
plt.subplot(1, 2, 2)
plt.scatter(y_test, y_test_pred, alpha=0.5, edgecolors='k', linewidth=0.5, color='orange')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel('Precio Real (miles de $)')
plt.ylabel('Precio Predicho (miles de $)')
plt.title(f'Conjunto de Prueba\nR² = {test_r2:.4f}')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de residuos
residuos = y_test - y_test_pred

plt.figure(figsize=(12, 5))

# Histograma de residuos
plt.subplot(1, 2, 1)
plt.hist(residuos, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Residuos (Error de Predicción)')
plt.ylabel('Frecuencia')
plt.title('Distribución de Residuos')
plt.axvline(x=0, color='red', linestyle='--', linewidth=2)

# Residuos vs predicciones
plt.subplot(1, 2, 2)
plt.scatter(y_test_pred, residuos, alpha=0.5, edgecolors='k', linewidth=0.5)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2)
plt.xlabel('Precio Predicho (miles de $)')
plt.ylabel('Residuos')
plt.title('Residuos vs Predicciones')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Importancia de las Características

In [ ]:
# Visualizar los coeficientes (importancia de características)
coef_df = pd.DataFrame({
    'Característica': X.columns,
    'Coeficiente': lr_model.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

print("Coeficientes del modelo (ordenados por magnitud):")
print(coef_df)

# Gráfico de barras
plt.figure(figsize=(10, 6))
colors = ['red' if x < 0 else 'green' for x in coef_df['Coeficiente']]
plt.barh(coef_df['Característica'], coef_df['Coeficiente'], color=colors, alpha=0.7)
plt.xlabel('Coeficiente')
plt.title('Impacto de Características en el Precio\n(Verde: Aumenta precio, Rojo: Disminuye precio)')
plt.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.grid(alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 8. Validación Cruzada

In [ ]:
# Realizar validación cruzada con 5 pliegues
cv_scores = cross_val_score(lr_model, X, y, cv=5, scoring='r2')

print("Scores de Validación Cruzada (R²):")
print(cv_scores)
print(f"\nMedia: {cv_scores.mean():.4f}")
print(f"Desviación estándar: {cv_scores.std():.4f}")

# Visualización
plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), cv_scores, color='steelblue', alpha=0.7, edgecolor='black')
plt.axhline(y=cv_scores.mean(), color='red', linestyle='--', 
            label=f'Media: {cv_scores.mean():.4f}')
plt.xlabel('Fold')
plt.ylabel('R² Score')
plt.title('Scores de Validación Cruzada')
plt.legend()
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 9. Ejemplo de Predicción con Nuevos Datos

In [ ]:
# Crear una nueva casa de ejemplo para predecir su precio
nueva_casa = pd.DataFrame({
    'CRIM': [0.5],        # Baja criminalidad
    'RM': [7.0],          # 7 habitaciones
    'AGE': [30.0],        # Casa relativamente nueva
    'DIS': [3.0],         # Distancia moderada
    'TAX': [300.0],       # Impuestos bajos
    'PTRATIO': [15.0],    # Buen ratio alumno-maestro
    'LSTAT': [5.0]        # Bajo % de población de bajo estatus
})

precio_predicho = lr_model.predict(nueva_casa)[0]

print("Características de la nueva casa:")
print(nueva_casa.T)
print(f"\nPrecio predicho: ${precio_predicho:.2f}k")

## Conclusiones

- La regresión lineal es un modelo simple pero poderoso para predecir valores continuos
- El R² indica qué porcentaje de la varianza de los datos es explicado por el modelo
- Los coeficientes muestran el impacto de cada característica en la predicción
- Es importante verificar que los residuos se distribuyan normalmente alrededor de cero
- La validación cruzada ayuda a evaluar la estabilidad del modelo